# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library, following best practices for referencing each dataset entity by its `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Accessing properties from dataset.metadata
meta = dataset.metadata
print(f"Dataset title: {meta.name}\nDescription: {meta.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We'll print out all record set `@id`s, field `@id`s, and column `@id`s from the dataset's Croissant schema.


In [ ]:
# Helper: Print all record sets, fields (by @id), and columns in the dataset
record_sets = meta.record_sets

print('Available Record Sets:')
for rs in record_sets:
    print(f"  - RecordSet @id: {rs.id} (name: {rs.name if hasattr(rs,'name') else 'N/A'})")
    if hasattr(rs, 'fields') and rs.fields:
        for field in rs.fields:
            print(f"    - Field @id: {field.id} (name: {field.name if hasattr(field,'name') else 'N/A'})")
            if hasattr(field, 'columns'):
                for col in field.columns:
                    print(f"      - Column @id: {col.id} (name: {col.name if hasattr(col,'name') else 'N/A'})")

Let's print out a sample of the records from the first record set using its `@id`.

In [ ]:
# Print 2 records from the first record set by @id (for structure preview)
if record_sets:
    first_record_set_id = record_sets[0].id
    print(f'First RecordSet @id: {first_record_set_id}')
    sample = list(dataset.records(record_set=first_record_set_id))[:2]
    for idx, rec in enumerate(sample):
        print(f"Sample record {idx}: {rec}")
else:
    print('No record sets defined in the schema.')

## 3. Data Extraction
Load data from one or more record sets into pandas DataFrames for analysis. All record set and field references will use their `@id`.


In [ ]:
# Collect all record set @id values
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    all_records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(all_records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for RecordSet @id: {record_set_id}, shape: {df.shape}")

# For demonstration, print the columns and first rows for the first record set
if record_set_ids:
    demo_id = record_set_ids[0]
    print(f"Columns for {demo_id}:")
    print(dataframes[demo_id].columns.tolist())
    display(dataframes[demo_id].head())

## 4. Exploratory Data Analysis (EDA)
We will pick a numeric field for analysis. Fields and columns are always referenced by their `@id`s. (Refer to Section 2 for valid values.)

- We'll filter records with a numeric field above a threshold
- Normalize the selected numeric field
- Group by a categorical field


In [ ]:
# Example usage: Update these IDs to match those found in section 2

record_set_id = record_set_ids[0]  # Use the first record set

# Find a numeric field (by id) in the first record set
fields = None
for rs in record_sets:
    if rs.id == record_set_id:
        fields = rs.fields or []
        break
numeric_field_id = None
group_field_id = None
# Try to pick reasonable default field names
for field in fields:
    if hasattr(field, 'data_type') and field.data_type in ('schema:Float', 'schema:Number', 'schema:Integer') and numeric_field_id is None:
        numeric_field_id = field.id  # as per Croissant, @id is the field column name
    if hasattr(field, 'data_type') and field.data_type in ('schema:Text') and group_field_id is None:
        group_field_id = field.id
# If not found, pick any field
df = dataframes[record_set_id]
if numeric_field_id is None:
    # Fallback, use first float-ish column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
if group_field_id is None:
    # Fallback, any non-numeric column
    for col in df.columns:
        if not pd.api.types.is_numeric_dtype(df[col]):
            group_field_id = col
            break

# EDA steps
if numeric_field_id is None or numeric_field_id not in df.columns:
    print('No numeric field found for EDA.')
else:
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df[[numeric_field_id]].head())
    
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field, and optionally, the grouped means.
All field references are by their Croissant `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id is not None and group_field_id in df.columns:
        top_groups = df[group_field_id].value_counts().head(5).index
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df[df[group_field_id].isin(top_groups)])
        plt.title(f"{numeric_field_id} distribution by {group_field_id} (Top 5 groups)")
        plt.show()
else:
    print('No numeric field available for visualization.')

## 6. Conclusion

In this notebook, we loaded and explored the FAIR^2 dataset, referenced all entities by their Croissant `@id`, previewed record sets and their fields, and performed basic exploratory data analysis and visualization. For robust or publication-level analyses, always refer to original field definitions and consider full domain context from the Croissant schema's metadata.